<h1>Chapter 7 - Advanced Text Generation Techniques and Tools</h1>
<i>Going beyond prompt engineering.</i>

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter07/Chapter%207%20-%20Advanced%20Text%20Generation%20Techniques%20and%20Tools.ipynb)

---

This notebook is for Chapter 7 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>

### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [1]:
# %%capture
# !pip install langchain>=0.1.17 openai>=1.13.3 langchain_openai>=0.1.6 transformers>=4.40.1 datasets>=2.18.0 accelerate>=0.27.2 sentence-transformers>=2.5.1 duckduckgo-search>=5.2.2 langchain_community
# !CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python==0.2.69

# Loading an LLM

## Dos formas diferentes de cargar un modelo

### GGUF + llama.cpp + LangChain

```text
Hugging Face
      │
      ▼
descargar archivo .gguf
      │
      ▼
archivo local
      │
      ▼
LlamaCpp
      │
      ▼
LangChain
```

Aquí necesitan tener físicamente el archivo `.gguf` en disco y pasar su ruta:

```python
model_path="Phi-3-mini-4k-instruct-fp16.gguf"
```

---

### Transformers + PyTorch

Con `from_pretrained()` el flujo habitual sería:

```text
Hugging Face repo
      │
      ▼
transformers
      │
      ▼
AutoModelForCausalLM.from_pretrained(...)
      │
      ▼
modelo PyTorch
```

Por tanto, **`from_pretrained()` no es una forma universal de cargar cualquier modelo de Hugging Face**.

Es una forma típica de cargar modelos compatibles con la API de `transformers`.

En esta sección quieren enseñar otra vía:

```text
GGUF + llama.cpp + LangChain
```

In [2]:
from pathlib import Path
import wget

model_path = Path(r"D:\AI\HuggingFace\Phi-3-mini-4k-instruct-fp16.gguf")

url = "https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf"

if model_path.exists():
    print("El modelo ya está descargado.")
else:
    print("Descargando modelo...")
    wget.download(url, str(model_path))
    print("\nDescarga completada.")


# Hago esta lógica para que no descargue el modelo con wget de nuevo ya que cambié el fichero a otra carpeta    
# If this command does not work for you, you can use the link directly to download the model
# https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

El modelo ya está descargado.


## Versión alternativa: Descargar el archivo GGUF con Hugging Face Hub

También podría haberse descargado directamente desde Python utilizando `huggingface_hub`:

```python
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="microsoft/Phi-3-mini-4k-instruct-gguf",
    filename="Phi-3-mini-4k-instruct-fp16.gguf"
)
```

El flujo sería:

```text
Hugging Face Hub
       │
       ▼
hf_hub_download(...)
       │
       ▼
descarga el archivo .gguf
       │
       ▼
lo guarda en la caché local
       │
       ▼
model_path
(ruta local al archivo)
       │
       ▼
LlamaCpp(
    model_path=model_path
)
       │
       ▼
LangChain
```

La diferencia es que `hf_hub_download()` **solo se encarga de descargar/localizar el archivo**.

No carga el modelo para inferencia:

```text
hf_hub_download()
        │
        └── descarga el .gguf

LlamaCpp
        │
        └── carga y ejecuta el .gguf
```

In [3]:
# Muevo el modelo de carpeta actual a carpeta de HF

#import shutil

#shutil.move(
#    "Phi-3-mini-4k-instruct-fp16.gguf",
#    r"D:\AI\HuggingFace\Phi-3-mini-4k-instruct-fp16.gguf"
#)


In [4]:
# Compruebo que está ahí
import os

os.path.exists(r"D:\AI\HuggingFace\Phi-3-mini-4k-instruct-fp16.gguf")

True

In [5]:
from langchain import LlamaCpp

# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path=r"D:\AI\HuggingFace\Phi-3-mini-4k-instruct-fp16.gguf", # dónde está el modelo físicamente
    n_gpu_layers=-1, # intenta poner todas las capas posibles en la GPU
    max_tokens=500, # como máximo genera 500 tokens de salida
    n_ctx=2048, # ventana de contexto de 2048 tokens
    seed=42, # fija una semilla para favorecer reproducibilidad
    verbose=False # no muestra toda la información interna de llama.cpp.
)

# Aquí estamos pasando de «tener el archivo del modelo» a cargarlo para poder hacer inferencia con él.
# llm pasa a ser un objeto de LangChain que te permite enviar prompts a tu Phi-3 local

In [6]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

# .invoke(...) → método de LangChain para ejecutar el modelo con una entrada.
# "Hi! My name..." → el prompt que le mandas.

''

### Chains

In [7]:
from langchain import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = """<|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)
# Tengo esta plantilla y existe un hueco llamado "input_prompt" que rellenaré posteriormente

In [8]:
basic_chain = prompt | llm
# encadenamos 2 componentes prompt y LLM
# ahora input -> PromptTemplate -> LLM -> Output

In [9]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

# Cuando invoco a la cadena hay que decirle qué valor corresponde a esa variable
# El diccionario hace esa asociación

" Hello Maarten! The answer to 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another, resulting in two units.\n\n---"

### Multiple Chains

In [10]:
from langchain import LLMChain

# Create a chain for the title of our story
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title") # crea una minicadena independiente
# tenemos ahora 2 cosas:
    # PromptTemplate + LLM
    # output_key = title

C:\Users\srmjf\AppData\Local\Temp\ipykernel_31276\3838147951.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 0.3.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title") # crea una minicadena independiente


In [11]:
title.invoke({"summary": "a girl that lost her mother"})

C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\llama_cpp\llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': ' "Echoes of a Mother\'s Love: A Journey through Grief"'}

Ahora vemos summary + Title

In [12]:
# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character") # crea otra minicadena independiente acumulando la variable character

In [13]:
character.invoke({
    "summary": "a girl that lost her mother",
    "title": "Whispers of Loss: A Journey Through Grief"
})

C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\llama_cpp\llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': 'Whispers of Loss: A Journey Through Grief',
 'character': ' The protagonist, young Emily, is an introspective and sensitive ten-year-old who has recently experienced profound grief following the loss of her beloved mother. Her journey through sorrow is characterized by a quiet determination to understand and process her emotions, ultimately transforming into resilience that will guide her forward in life.'}

In [14]:
# Create a chain for the story using the summary, title, and character description
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main charachter is: {character}. Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [15]:
story.invoke({
    "summary": "a girl that lost her mother",
    "title": "Whispers of Loss: A Journey Through Grief",
    "character": "A young girl struggling to cope with the loss of her mother..."
})

C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\llama_cpp\llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': 'Whispers of Loss: A Journey Through Grief',
 'character': 'A young girl struggling to cope with the loss of her mother...',
 'story': ' In the heart-wrenching tale "Whispers of Loss: A Journey Through Grief," a spirited young girl, Emma, grapples with an unfathomable reality as she navigates life after losing her mother. Each day presents a labyrinth of sorrow and memories that threaten to consume her tender soul. Amidst the overwhelming silence left by her mother\'s departure, Emma embarks on an emotional odyssey, seeking solace in nature\'s embrace, finding strength within herself through cherished moments shared with her beloved matriarch. In this delicate narrative of resilience and hope, Emma discovers that even amidst the darkest whispers of loss, a heartfelt connection to her mother remains immortalized in the legacy she leaves behind - guiding her onwards through grief\'s shadowed path with unwavering determination and love.

Hasta aquí cada cadena es independiente una de otra

In [16]:
# Combine all three components to create the full chain
llm_chain = title | character | story

# Con esto creamos una cadena única
# Ventaja -> Cuando el problema se vuelve complejo, controla cada etapa, puede cambair una, inspeccionar resultados intermedios y
# hacer que una salida alimente la siguiente

In [17]:
llm_chain.invoke("a girl that lost her mother")

C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\llama_cpp\llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\llama_cpp\llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\llama_cpp\llama.py:1031: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': ' "The Echoes of Mother\'s Absence"',
 'character': ' The central character in "The Echoes of Mother\'s Absence" is a resilient young girl, Isabella, who struggles to cope with her mother\'s sudden and tragic passing while navigating the complexities of grief and self-discovery. Bound by love but propelled forward by loss, she embarks on an emotional journey towards healing that reveals both the profundity of her connection to her mother and the strength within herself to move beyond it.',
 'story': ' "The Echoes of Mother\'s Absence" traces Isabella\'s heart-wrenching odyssey through grief after losing her beloved mother in a tragic accident. As she grapples with overwhelming sorrow, Isabella finds herself drawn to the melodies of their cherished lullabies and the comforting embrace of childhood memories. Seeking solace amidst the cacophony of her anguish, she discovers a newfound resilience within herself, propelled by an unwaverin

# Memory

In [18]:
# Let's give the LLM our name
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

" Hello, Maarten! The answer to 1 + 1 is 2. It's a basic arithmetic operation where you combine one unit with another, resulting in two units total."

In [19]:
# Next, we ask the LLM to reproduce the name
basic_chain.invoke({"input_prompt": "What is my name?"})

" I'm unable to determine your name without additional context. As an AI, I don't have the ability to access personal data unless it has been shared with me in the course of our conversation for the purpose of assisting you. If you're looking to find out more about yourself or require assistance that may involve personal information, consider checking privacy-compliant methods or platforms designed for self-discovery."

## ConversationBuffer

In [20]:
# Create an updated prompt template to include a chat history
template = """<|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [21]:
from langchain.memory import ConversationBufferMemory

# Define the type of Memory we will use
memory = ConversationBufferMemory(memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [22]:
# Generate a conversation and ask a basic question
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

{'input_prompt': 'Hi! My name is Maarten. What is 1 + 1?',
 'chat_history': '',
 'text': " The answer to 1 + 1 is 2. It's a basic arithmetic operation where you combine two units together, resulting in two units in total.\n\nHere's the breakdown of the calculation:\n1 (unit) + 1 (another unit) = 2 (units combined)."}

In [23]:
# Does the LLM remember the name we gave it?
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten. What is 1 + 1?\nAI:  The answer to 1 + 1 is 2. It's a basic arithmetic operation where you combine two units together, resulting in two units in total.\n\nHere's the breakdown of the calculation:\n1 (unit) + 1 (another unit) = 2 (units combined).",
 'text': ' My name is Assistant. Nice to meet you, Maarten! It\'s great that you know your basic arithmetic operations. The answer indeed remains 2 for the calculation of 1 + 1.\n\nAs for your question about my name, I am called "Assistant." How may I further assist you?'}

## ConversationBufferMemoryWindow

In [24]:
from langchain.memory import ConversationBufferWindowMemory

# Retain only the last 2 conversations in memory
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [25]:
# Ask two questions and generate two conversations in its memory
llm_chain.invoke({"input_prompt":"Hi! My name is Maarten and I am 33 years old. What is 1 + 1?"})
llm_chain.invoke({"input_prompt":"What is 3 + 3?"})

{'input_prompt': 'What is 3 + 3?',
 'chat_history': 'Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  The answer to the mathematical question "What is 1 + 1?" is 2. As for your introduction, it\'s nice to meet you, Maarten! Here we can discuss various topics if you wish. Just let me know what interests you or anything specific you have in mind.\n\nHowever, based on the context provided, there doesn\'t seem to be a complex question related to mathematics since 1 + 1 is quite straightforward. If you have any other questions about math or different subjects, feel free to ask!',
 'text': ' The answer to the mathematical question "What is 3 + 3?" is 6. It\'s great to meet you as well, Maarten! I\'m here to help with any questions or topics you have in mind. Whether it\'s math, science, technology, or something else, just let me know what you need assistance with.'}

In [26]:
# Check whether it knows the name we gave it
llm_chain.invoke({"input_prompt":"What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': 'Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  The answer to the mathematical question "What is 1 + 1?" is 2. As for your introduction, it\'s nice to meet you, Maarten! Here we can discuss various topics if you wish. Just let me know what interests you or anything specific you have in mind.\n\nHowever, based on the context provided, there doesn\'t seem to be a complex question related to mathematics since 1 + 1 is quite straightforward. If you have any other questions about math or different subjects, feel free to ask!\nHuman: What is 3 + 3?\nAI:  The answer to the mathematical question "What is 3 + 3?" is 6. It\'s great to meet you as well, Maarten! I\'m here to help with any questions or topics you have in mind. Whether it\'s math, science, technology, or something else, just let me know what you need assistance with.',
 'text': " Your name is Maarten, as you introduced yourself at the beginning of our 

In [27]:
# Check whether it knows the age we gave it
llm_chain.invoke({"input_prompt":"What is my age?"})

{'input_prompt': 'What is my age?',
 'chat_history': 'Human: What is 3 + 3?\nAI:  The answer to the mathematical question "What is 3 + 3?" is 6. It\'s great to meet you as well, Maarten! I\'m here to help with any questions or topics you have in mind. Whether it\'s math, science, technology, or something else, just let me know what you need assistance with.\nHuman: What is my name?\nAI:  Your name is Maarten, as you introduced yourself at the beginning of our conversation. If there\'s anything specific you would like to discuss or any other questions you have in mind, feel free to ask!',
 'text': " I'm unable to determine your age without additional information. Age privacy guidelines prevent me from accessing or storing personal data such as your birthdate. If you need help with calculating ages based on certain criteria, please provide the relevant details and context."}

Lógico porque pusimos que recordara hasta dos conversaciones y la edad se mecionó hace tres conversaciones

## ConversationSummary

In [28]:
# Create a summary prompt template
summary_prompt_template = """<|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

In [29]:
from langchain.memory import ConversationSummaryMemory

# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm, # puede ser otro LLM más pequeño
    memory_key="chat_history",
    prompt=summary_prompt # prompt que gestiona la memoria
)


# ConversationSummaryMemory ya está programada internamente para trabajar precisamente con esas dos variables:
    # summary → el resumen acumulado hasta ese momento.
    # new_lines → las nuevas intervenciones que todavía no se han incorporado al resumen.



# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt, # user prompt
    llm=llm,
    memory=memory # que está gestionada por summary_prompt
)

In [30]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': ' Maarten introduces himself and asks the AI for the result of the simple addition problem "1 + 1". The AI responds by confirming that the answer is 2, explaining it as a basic arithmetic operation where one unit is added to another, yielding two units in total.\n\nNew summary: Maarten inquires about the sum of 1+1 and receives an explanation of addition resulting in 2 from the AI.',
 'text': ' Your name has not been mentioned in the current conversation; you referred to "Maarten" as a participant, but did not provide your own name.'}

In [31]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

{'input_prompt': 'What was the first question I asked?',
 'chat_history': ' Maarten asks for the sum of 1+1 and gets an explanation that addition equals 2 from the AI, while also clarifying that his actual name was not revealed in the conversation. The AI points out that only "Maarten" has been mentioned so far.',
 'text': ' The first question you asked was: "Can you tell me what 1+1 equals?"'}

In [32]:
# Check what the summary is thus far
memory.load_memory_variables({})

{'chat_history': ' Maarten inquires about the sum of 1+1, receiving an explanation that addition equals 2 from the AI. However, their actual name remains unrevealed in the conversation as only "Maarten" has been mentioned so far. The human then asks what the first question they asked was, and the AI confirms it was to determine the result of 1+1.'}

# Agents

Si haces:

```python
import os
os.environ["OPENAI_API_KEY"] = "MY_KEY"
```

y subes el código a GitHub, **tu key será visible**.

Mejor:

```text
.env → guarda la key
.gitignore → evita subir .env
```

```python
from dotenv import load_dotenv

load_dotenv()
```

Así `ChatOpenAI` puede encontrar `OPENAI_API_KEY` sin escribirla en el código.

In [35]:
# Mejor así

from dotenv import load_dotenv 
from langchain_openai import ChatOpenAI

load_dotenv() 
# lee mi archi .env con OPENAI_API_KEY=sk-proj-xxxxxxxxxxxx y mete ese valor en las variables de entorno del
# proceso de Python
# ChatOpenAI está programado para buscar automáticamente una variable de entorno llamada OPENAI_API_KEY

openai_llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    temperature=0
)

In [36]:
# Create the ReAct template
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

# {agent_scratchpad} es el historial de trabajo del agente durante la ejecución de ReAct.

In [43]:
from langchain.agents import load_tools, Tool
from ddgs import DDGS # alternativa buscador a duckduckgo
# You can create the tool to pass to an agent

def web_search(query):
    results = DDGS().text(
        query,
        max_results=5,
        backend="auto"
    )
    return str(results)




search = DuckDuckGoSearchResults()
search_tool = Tool(
    name="web_search",
    description="Search the web for current information.",
    func=web_search
)

# Prepare tools
tools = load_tools(["llm-math"], llm=openai_llm)
tools.append(search_tool) # añado search_tool a tool

In [44]:
# Creamos el agente ReAct y el agente ejecutor

from langchain.agents import AgentExecutor, create_react_agent

# Construct the ReAct agent
agent = create_react_agent(openai_llm, tools, prompt)
# agente ReAct -> Sabe qué LLM usar, qué herramientas existen y qué formato/promt debe seguir

agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)
# el agente ejecutor es el que realiza el bucle

In [45]:
# What is the Price of a MacBook Pro?
agent_executor.invoke(
    {
        "input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?"
    }
)



> Entering new AgentExecutor chain...
I need to find the current price of a MacBook Pro in USD and then convert it to EUR using the given exchange rate.
Action: web_search
Action Input: "current price of MacBook Pro in USD"[{'title': 'MacBook Pro in United States | USD Prices – The Mac Index', 'href': 'https://themacindex.com/us/products/macbook-pro', 'body': "1 month ago - Mexico offers up to 9% tax refund, bringing the effective price down to $2,451. Regional price differences for the MacBook Pro 14-inch M5 10-Core 16GB 1TB are notable: ... Take note that we don't have prices for India yet due to technical ..."}, {'title': 'Apple MacBook Pro price comparison 2026| Statista', 'href': 'https://www.statista.com/statistics/1360426/apple-macbook-pro-price-comparison/', 'body': 'February 12, 2026 - As of February 2026, the latest MacBook Pro 14-inch M4 model started with a retail price of ***** U.S.'}, {'title': 'Mac Computers 2026 Best Sale Price Deals & Discounts', 'href': 'https://pri

{'input': 'What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?',
 'output': 'Agent stopped due to iteration limit or time limit.'}